# Ch 7: Finetuning using Instruction Dataset 

In [1]:
import json
file_path="Dataset\instruction-data.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)
print("Number of entries:", len(data))

Number of entries: 1100


<>:2: SyntaxWarning: invalid escape sequence '\i'
<>:2: SyntaxWarning: invalid escape sequence '\i'
C:\Users\subra\AppData\Local\Temp\ipykernel_23460\3204568704.py:2: SyntaxWarning: invalid escape sequence '\i'
  file_path="Dataset\instruction-data.json"


In [2]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


- There are 2 common ways for formatting instruction:
1. Alpaca(Instruction,Input,Response):https://crfm.stanford.edu/2023/03/13/alpaca.html
2. Phi3(Instruction,Response):https://arxiv.org/abs/2404.14219
- In this chapter, we use Alpaca-style prompt formatting, which was the original prompt template for instruction finetuning

In [3]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

In [4]:
# with input sectiopn
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [5]:
# without input section
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


- We are dividing the dataset into traning(85%),testing(10%) and validation(5%)

In [6]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


## Preparing Dataset

In [7]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [8]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


- We will pad the extra spaces using `<|endoftext|>` token to adjust all the inputs to same size using a collate function.
- Tasks done by collate function:
    1. Padding items with `<|endoftext|>`(Each item must have atleast one `<|endoftext|>`)
    2. Creating input and target batches
    3. Replacing padding tokens with ignore_index 
    4. Truncating the inputs and to maximum length allowed(optional)

In [9]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch1=[inputs_1,inputs_2,inputs_3]
print(batch1)
batch=(
    inputs_1,
    inputs_2,
    inputs_3
)
print(batch)

[[0, 1, 2, 3, 4], [5, 6], [7, 8, 9]]
([0, 1, 2, 3, 4], [5, 6], [7, 8, 9])


In [10]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        # Padding sentences
        # Adding one extra pad token for target batch
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # Creating input and target batches
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # Replacing padding tokens with ignore_index
        # Ignore the first pad token
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
        # Truncating the inputs and to maximum length allowed(optional)
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [11]:
print(custom_collate_fn(batch1)[0])
print(custom_collate_fn(batch1)[1])

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [13]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [14]:

from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [15]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [16]:
print("Train loader:")
count=0
for inputs, targets in train_loader:
    count=count+1
print(count)

Train loader:
116


In [17]:
1100//8*0.85

116.45

In [18]:

print(inputs[0])

tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,
          985,   576,    13,   198,   198, 21017, 23412,    25,   198,   464,
         5156,   318,   845, 13779,    13,   198,   198, 21017, 18261,    25,
          198,   464,  5156,   318,   355, 13779,   355,   257,  4936,    13,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
       device='cuda:0')


In [19]:

print(targets[0])

tensor([  318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,   257,
         2882,   326, 20431, 32543,   262,  2581,    13,   198,   198, 21017,
        46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,   985,
          576,    13,   198,   198, 21017, 23412,    25,   198,   464,  5156,
          318,   845, 13779,    13,   198,   198, 21017, 18261,    25,   198,
          464,  5156,   318,   355, 13779,   355,   257,  4936,    13, 50256,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
       device='cuda:0')


In [20]:
from GPTModules import GPTModel,GPT2_CONFIG

In [21]:
model = GPTModel(GPT2_CONFIG)
model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))

<All keys matched successfully>

In [22]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [23]:
from GPTModules import generate,text_to_token_ids,token_ids_to_text
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=GPT2_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

In [24]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)
print(response_text)

The process by the sentence is a question.


In [25]:
from GPTModules import calculate_dataloader_loss
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss=calculate_dataloader_loss(train_loader,model,device)
    val_loss=calculate_dataloader_loss(val_loader,model,device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: tensor(1.4257, device='cuda:0')
Validation loss: tensor(2.0272, device='cuda:0')


In [26]:
import subprocess

def show_gpu_memory():
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,nounits,noheader']
        )
        gpus = result.decode().strip().split('\n')
        for idx, gpu in enumerate(gpus):
            used, total = map(int, gpu.split(','))
            print(f"GPU {idx}: {used} MB / {total} MB used ({(used/total)*100:.1f}%)")
    except FileNotFoundError:
        print("nvidia-smi not found. Make sure NVIDIA drivers are installed and in your PATH.")
    except Exception as e:
        print(f"Error while checking GPU memory: {e}")

# Call the function
show_gpu_memory()

GPU 0: 1997 MB / 4096 MB used (48.8%)


In [27]:
import gc

# Delete any variables (optional but recommended)
# del your_tensor  # example if you have specific variables

# Run garbage collection to destroy any lingering tensors
gc.collect()

# Empty PyTorch's CUDA memory cache
torch.cuda.empty_cache()

show_gpu_memory()

GPU 0: 769 MB / 4096 MB used (18.8%)


In [ ]:
from GPTModules import train_model
train_losses,val_losses,epochs=train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=5,
    start_text=format_input(val_data[0]),
    tokenizer=tokenizer,
    new_tokens=50,
    learning_rate=0.00005,
    device=device
)

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model = GPTModel(GPT2_CONFIG)
model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))

In [28]:
torch.manual_seed(123)
for entry in test_data[:3]:
    input_text=format_input(entry)
    token_ids=generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=GPT2_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.

Model response:
>> The process by which water is 'I am.
-------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?

Correct response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The chemical formula for 'The process by which water is 'I am degrees Celsius.
-------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Name the author of 'Pride and Prejudice'.

Correct response:
>> Jane Austen.

Model response:
>> The process b

In [29]:
!pip install tqdm

In [30]:

from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=GPT2_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

    test_data[i]["model_response"] = response_text


with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  # "indent" for pretty-printing

100%|██████████| 110/110 [00:49<00:00,  2.23it/s]


In [31]:
!pip install psutil

In [33]:
!ollama run llama3

'ollama' is not recognized as an internal or external command,
operable program or batch file.


In [32]:

import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

RuntimeError: Ollama not running. Launch ollama before proceeding.